## Combined Power Plant Model Quality Monitoring (Artifical Ground Truth Testing Alarm)

This model quality monitoring with artifical ground truth for testing alarm was created following AAI 540 Lab 5


### 1. Setup

#### 1.1 Install Package and Import necessary libraries


In [195]:
#!pip install -q --upgrade pip

In [194]:
#!pip3 show sagemaker

In [62]:
%time
from datetime import datetime, timedelta, timezone
import json
import os
import re
import boto3
from time import sleep
from threading import Thread

import pandas as pd



CPU times: user 4 μs, sys: 0 ns, total: 4 μs
Wall time: 8.58 μs


##### Change the working dir for the code

In [63]:
%cd ~/MSAAI540/

/home/sagemaker-user/MSAAI540


In [67]:
from sagemaker import get_execution_role, session, Session, image_uris
from sagemaker.s3 import S3Downloader, S3Uploader
from sagemaker.processing import ProcessingJob
from sagemaker.serializers import CSVSerializer

from sagemaker.model import Model
from sagemaker.model_monitor import DataCaptureConfig

session = Session()

#### 1.2 AWS region and  IAM Role

In [68]:
# Get Execution role
role = get_execution_role()
print("RoleArn:", role)

region = session.boto_region_name
print("Region:", region)

RoleArn: arn:aws:iam::582544415612:role/LabRole
Region: us-east-1


#### 1.3 S3 bucket and prefixes

In [70]:
# Setup S3 bucket
bucket = session.default_bucket()
print("Bucket:", bucket)
prefix = "sagemaker/XGB-ModelQualityMonitor-20240225test"

##S3 prefixes
data_capture_prefix = f"{prefix}/datacapture"
s3_capture_upload_path = f"s3://{bucket}/{data_capture_prefix}"

ground_truth_upload_path = (
    f"s3://{bucket}/{prefix}/ground_truth_data/{datetime.now():%Y-%m-%d-%H-%M-%S}"
)

reports_prefix = f"{prefix}/reports"
s3_report_path = f"s3://{bucket}/{reports_prefix}"

##Get the model monitor image
monitor_image_uri = image_uris.retrieve(framework="model-monitor", region=region)

print("Image URI:", monitor_image_uri)
print(f"Capture path: {s3_capture_upload_path}")
print(f"Ground truth path: {ground_truth_upload_path}")
print(f"Report path: {s3_report_path}")

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


Bucket: sagemaker-us-east-1-582544415612
Image URI: 156813124566.dkr.ecr.us-east-1.amazonaws.com/sagemaker-model-monitor-analyzer
Capture path: s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/datacapture
Ground truth path: s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/ground_truth_data/2026-02-22-01-46-50
Report path: s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/reports


### 2. Deploy pre-trained model with data capture enabled <a id='deploy'></a>

#### 2.1 Create SageMaker Model entity

In [71]:
import sagemaker
# Retrieve the inference docker container uri
deploy_image_uri = image = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=boto3.Session().region_name,
    version="1.7-1"
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [72]:
model_name = f"ccpp-xgb-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
# url of Pretrained model
model_url = 's3://sagemaker-us-east-1-582544415612/CCPP-enery-prediction-xgb-regression/output/xgb-2026-02-18-06-55-12/xgb-2026-02-18-06-55-12/output/model.tar.gz'

model = Model(image_uri=deploy_image_uri, model_data=model_url, role=role, sagemaker_session=session)

/tmp/ipykernel_89118/3532243873.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  model_name = f"ccpp-xgb-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"


#### 2.2 Deploy the model with data capture enabled.

In [73]:
endpoint_name = f"ccpp-xgb-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
print("EndpointName =", endpoint_name)
JsonContentTypes = "application/json"
data_capture_config = DataCaptureConfig(
    enable_capture=True, sampling_percentage=100, destination_s3_uri=s3_capture_upload_path
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    entry_point="inference.py",
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config,
)

/tmp/ipykernel_89118/962745670.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  endpoint_name = f"ccpp-xgb-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"


EndpointName = ccpp-xgb-model-monitor-2026-02-22-0147


INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-02-22-01-47-01-681
INFO:sagemaker:Creating endpoint-config with name ccpp-xgb-model-monitor-2026-02-22-0147
INFO:sagemaker:Creating endpoint with name ccpp-xgb-model-monitor-2026-02-22-0147


------!

#### 2.3 Create the SageMaker Predictor object from the endpoint to be used for invoking the model

In [74]:
from sagemaker.predictor import Predictor
predictor = Predictor(
    endpoint_name=endpoint_name, sagemaker_session=session, serializer=CSVSerializer()
)

### 3. Generate a baseline for model quality performance <a id='generate-baseline'></a>

#### 3.1 Execute predictions using the validation dataset. 

In [75]:
validate_dataset = "validation_with_predictions.csv"

In [76]:
limit = 200  # Need at least 200 samples to compute standard deviations
i = 0
import json
with open(f"{curr_dir}/testdata/{validate_dataset}", "w") as baseline_file:
    baseline_file.write("prediction,label\n")  # our header
    with open(f"{curr_dir}/testdata/validation_data.csv", "r") as f:
        for row in f:
            (label, input_cols) = row.split(",", 1)
            prediction = predictor.predict(input_cols)
            data = json.loads(prediction)
            prediction_number = data
            baseline_file.write(f"{prediction_number},{label}\n")
            i += 1
            if i > limit:
                break
            print(".", end="", flush=True)
            sleep(0.5)
print()
print("Done!")

........................................................................................................................................................................................................
Done!


#### 3.2 Examine the predictions from the model

#### 3.3 Upload the predictions as a baseline dataset.

In [77]:
baseline_prefix = prefix + "/baselining"
baseline_data_prefix = baseline_prefix + "/data"
baseline_results_prefix = baseline_prefix + "/results"

baseline_data_uri = f"s3://{bucket}/{baseline_data_prefix}"
baseline_results_uri = f"s3://{bucket}/{baseline_results_prefix}"
print(f"Baseline data uri: {baseline_data_uri}")
print(f"Baseline results uri: {baseline_results_uri}")

Baseline data uri: s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/baselining/data
Baseline results uri: s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/baselining/results


In [78]:
baseline_dataset_uri = S3Uploader.upload(f"{curr_dir}/testdata/{validate_dataset}", baseline_data_uri)
baseline_dataset_uri

's3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/baselining/data/validation_with_predictions.csv'

#### 3.4 Create a baselining job with validation dataset predictions

In [79]:
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor import EndpointInput
from sagemaker.model_monitor.dataset_format import DatasetFormat

In [80]:
# Create the model quality monitoring object
xgb_model_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,
)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


#### 3.5 Explore the results of the baselining job

In [81]:
# Name of the model quality baseline job
baseline_job_name = f"ccpp-xgb-model-baseline-job-{datetime.utcnow():%Y-%m-%d-%H%M}"

/tmp/ipykernel_89118/1203395883.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  baseline_job_name = f"ccpp-xgb-model-baseline-job-{datetime.utcnow():%Y-%m-%d-%H%M}"


In [82]:
# Execute the baseline suggestion job.
job = xgb_model_quality_monitor.suggest_baseline(
    job_name=baseline_job_name,
    baseline_dataset=baseline_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_uri,
    problem_type="Regression",
    inference_attribute="prediction",
    ground_truth_attribute="label",
)


INFO:sagemaker:Creating processing-job with name ccpp-xgb-model-baseline-job-2026-02-22-0152


In [83]:
job.wait(logs=False)

...........................................................!

In [84]:
baseline_job = xgb_model_quality_monitor.latest_baselining_job

In [85]:
# Access the baseline statistics and extract regression metrics
regression_metrics = baseline_job.baseline_statistics().body_dict["regression_metrics"]

# Normalize the regression metrics into a pandas DataFrame for easier viewing
pd.json_normalize(regression_metrics).T

,0
mae.value,1.364941
mae.standard_deviation,0.020079
mse.value,2.721507
mse.standard_deviation,0.052868
rmse.value,1.649699
rmse.standard_deviation,0.016072
r2.value,0.986821
r2.standard_deviation,0.000317


In [86]:
# Access the suggested constraints and extract regression constraints
regression_constraints = baseline_job.suggested_constraints().body_dict["regression_constraints"]

# Convert the regression constraints into a pandas DataFrame and transpose it
pd.DataFrame(regression_constraints).T

,threshold,comparison_operator
mae,1.364941,GreaterThanThreshold
mse,2.721507,GreaterThanThreshold
rmse,1.649699,GreaterThanThreshold
r2,0.986821,LessThanThreshold


### 4. Setup continuous model monitoring to identify model quality drift <a id='analyze-model-quality-drift'></a>

#### 4.1 Generate prediction data for Model Quality  Monitoring

In [87]:
def invoke_endpoint(ep_name, file_name):
    with open(file_name, "r") as f:
        i = 0
        for row in f:
            payload = row.rstrip("\n")
            response = session.sagemaker_runtime_client.invoke_endpoint(
                EndpointName=endpoint_name,
                ContentType="text/csv",
                Body=payload,
                InferenceId=str(i),  # unique ID per row
            )["Body"].read()
            i += 1
            sleep(1)


def invoke_endpoint_forever():
    while True:
        try:
            invoke_endpoint(endpoint_name, f"{curr_dir}/testdata/batch_data.csv")
        except session.sagemaker_runtime_client.exceptions.ValidationError:
            pass


thread = Thread(target=invoke_endpoint_forever)
thread.start()

#### 4.2 View captured data

In [88]:
print("Waiting for captures to show up", end="")
for _ in range(120):
    capture_files = sorted(S3Downloader.list(f"{s3_capture_upload_path}/{endpoint_name}"))
    if capture_files:
        capture_file = S3Downloader.read_file(capture_files[-1]).split("\n")
        capture_record = json.loads(capture_file[0])
        if "inferenceId" in capture_record["eventMetadata"]:
            break
    print(".", end="", flush=True)
    sleep(1)
print()
print("Found Capture Files:")
print("\n ".join(capture_files[-3:]))

Waiting for captures to show up................................
Found Capture Files:
s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/datacapture/ccpp-xgb-model-monitor-2026-02-22-0147/AllTraffic/2026/02/22/01/50-34-257-42a03033-3095-45ce-8491-6e178f48c066.jsonl
 s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/datacapture/ccpp-xgb-model-monitor-2026-02-22-0147/AllTraffic/2026/02/22/01/51-34-631-323e3e21-b82d-4c3d-978a-81521d91f74f.jsonl
 s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/datacapture/ccpp-xgb-model-monitor-2026-02-22-0147/AllTraffic/2026/02/22/01/57-20-479-6832cfc2-8385-4d14-b174-3dcb0beacb17.jsonl


In [89]:
print("\n".join(capture_file[-3:-1]))

{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"13.21,30.429,1099.9,4.5634,648.3,2.3,6.0,1009.4,13.5,1012.0","encoding":"CSV"},"endpointOutput":{"observedContentType":"text/csv; charset=utf-8","mode":"OUTPUT","data":"149.01531982421875\n","encoding":"CSV"}},"eventMetadata":{"eventId":"875fd76e-4c6b-4cd5-b027-401cbc473369","inferenceId":"55","inferenceTime":"2026-02-22T01:58:19Z"},"eventVersion":"0"}
{"captureData":{"endpointInput":{"observedContentType":"text/csv","mode":"INPUT","data":"13.199,30.161,1100.0,4.5208,773.5,2.8,8.0,1009.2,38.3,1011.9","encoding":"CSV"},"endpointOutput":{"observedContentType":"text/csv; charset=utf-8","mode":"OUTPUT","data":"149.918701171875\n","encoding":"CSV"}},"eventMetadata":{"eventId":"7b612b22-0396-4646-8b20-bf7d44024d21","inferenceId":"56","inferenceTime":"2026-02-22T01:58:20Z"},"eventVersion":"0"}


In [90]:
print(json.dumps(capture_record, indent=2))

{
  "captureData": {
    "endpointInput": {
      "observedContentType": "text/csv",
      "mode": "INPUT",
      "data": "13.118,29.805,1099.9,4.3594,0.0,0.0,0.0,1010.2,0.0,1011.9",
      "encoding": "CSV"
    },
    "endpointOutput": {
      "observedContentType": "text/csv; charset=utf-8",
      "mode": "OUTPUT",
      "data": "148.51193237304688\n",
      "encoding": "CSV"
    }
  },
  "eventMetadata": {
    "eventId": "56a9a1e1-e7cd-4fa2-bcb8-b65e1bdfdf9c",
    "inferenceId": "0",
    "inferenceTime": "2026-02-22T01:57:20Z"
  },
  "eventVersion": "0"
}


Endpoint output from XGBoost is string value. This data need to be preprocess before the model monitoring schedule job can take only flat format data not array. Preprocess script has been created for the model monitoring.

#### 4.3 Generate synthetic ground truth with test_data

In [91]:
test_data = pd.read_csv(f"{curr_dir}/testdata/test_data.csv", header=None)
test_data

,0,1,2,3,4,5,6,7,8,9,10
0,147.21,13.118,29.805,1099.9,4.3594,0.0,0.0,0.0,1010.2,0.0,1011.9
1,147.14,13.160,30.018,1099.9,4.4136,0.0,0.0,0.0,1010.3,1.0,1011.9
2,147.66,13.204,30.199,1099.9,4.4296,0.0,0.0,0.0,1010.3,0.0,1012.0
3,147.14,13.121,29.979,1099.9,4.3978,19.9,0.1,0.0,1010.2,0.0,1012.0
4,146.55,13.094,29.712,1100.0,4.3754,144.2,0.5,1.0,1009.8,0.0,1010.5
...,...,...,...,...,...,...,...,...,...,...,...
2950,109.08,10.411,19.087,1037.0,3.1661,0.0,0.0,0.0,1028.5,78.2,1029.6
2951,108.79,10.344,19.016,1037.6,3.1923,0.0,0.0,0.0,1028.6,78.2,1028.8
2952,107.81,10.462,18.857,1038.0,3.3128,0.0,0.0,0.0,1028.5,46.0,1028.7
2953,131.41,11.771,23.563,1076.9,3.9831,0.0,0.0,0.0,1028.7,46.0,1028.7


The ground truth will have to be same format as capture data. Not same file type, but need to be same data type. As capture data is JSON file of prediction array. Ground truth do not need to be JSON, it can be CSV but need to be in array format to ensure that monitoring schecudule pre-process script can process it as same as capture data from model endpoint.

In [92]:
def ground_truth_with_id(inference_id):
    return {
        "groundTruthData": {
            "data": f"{(test_data[0][inference_id]-10.0)}",  # for testing set as constance 
            "encoding": "CSV",
        },
        "eventMetadata": {
            "eventId": str(inference_id),
        },
        "eventVersion": "0",
    }


def upload_ground_truth(records, upload_time):
    fake_records = [json.dumps(r) for r in records]
    data_to_upload = "\n".join(fake_records)
    target_s3_uri = f"{ground_truth_upload_path}/{upload_time:%Y/%m/%d/%H/%M%S}.jsonl"
    print(f"Uploading {len(fake_records)} records to", target_s3_uri)
    S3Uploader.upload_string_as_file_body(data_to_upload, target_s3_uri)

Ground truth data has been subtract 10.0 to trigger alarm

In [93]:
NUM_GROUND_TRUTH_RECORDS = 649  # 649 are the number of rows in data we're sending for inference


def generate_fake_ground_truth_forever():
    j = 0
    while True:
        fake_records = [ground_truth_with_id(i) for i in range(NUM_GROUND_TRUTH_RECORDS)]
        upload_ground_truth(fake_records, datetime.utcnow())
        j = (j + 1) % 5
        sleep(60 * 60)  # do this once an hour


gt_thread = Thread(target=generate_fake_ground_truth_forever)
gt_thread.start()

/tmp/ipykernel_89118/258847449.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  upload_ground_truth(fake_records, datetime.utcnow())


#### 4.4 Create a monitoring schedule

In [94]:
##Monitoring schedule name
smartgrid_monitor_schedule_name = (
    f"Xgb-model-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M}"
)

/tmp/ipykernel_89118/3864607400.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"Xgb-model-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M}"


Uploading 649 records to

In [139]:
# Create an enpointInput
endpointInput = EndpointInput(
    endpoint_name=predictor.endpoint_name,
    destination="/opt/ml/processing/input_data",
    inference_attribute="prediction" # use inference_attribute to identify the preiction value in Endpoint
)

In [178]:
s3_key = f"s3://{bucket}/{prefix}/preprocessor" #RTC:MSAAI540/src/code/pre_processor_handler.py
pre_processor_script = S3Uploader.upload( f"{curr_dir}/src/code/pre_processor_handler.py", s3_key)
pre_processor_script

's3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/preprocessor/pre_processor_handler.py'

pre_processor_script is the location of preprocessor.py store for the record_preprocessor_script to be used next step to deal with JSON array format at capture data at model Endpoint

In [179]:
# Create the monitoring schedule to execute every hour.
from sagemaker.model_monitor import CronExpressionGenerator
ccpp_monitor_schedule_name = (
    f"Xgb-model-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M%s}"
)
response = xgb_model_quality_monitor.create_monitoring_schedule(
    record_preprocessor_script=pre_processor_script, # call the preprocessor.py
    monitor_schedule_name=ccpp_monitor_schedule_name,
    endpoint_input=endpointInput,
    output_s3_uri=baseline_results_uri,
    problem_type="Regression",
    ground_truth_input=ground_truth_upload_path,
    constraints=baseline_job.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.now(),
    data_analysis_start_time="-PT1H",
    data_analysis_end_time="-PT0H",
    enable_cloudwatch_metrics=True,
)

/tmp/ipykernel_89118/710475260.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"Xgb-model-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M%s}"
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: Xgb-model-monitoring-schedule-2026-02-22-04521771735921


In [180]:
# Create the monitoring schedule
# monitoring schedule in the 'Scheduled' status
xgb_model_quality_monitor.describe_schedule()

{'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:582544415612:monitoring-schedule/Xgb-model-monitoring-schedule-2026-02-22-04521771735921',
 'MonitoringScheduleName': 'Xgb-model-monitoring-schedule-2026-02-22-04521771735921',
 'MonitoringScheduleStatus': 'Scheduled',
 'MonitoringType': 'ModelQuality',
 'CreationTime': datetime.datetime(2026, 2, 22, 4, 52, 2, 44000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 22, 4, 52, 6, 683000, tzinfo=tzlocal()),
 'MonitoringScheduleConfig': {'ScheduleConfig': {'ScheduleExpression': 'NOW',
   'DataAnalysisStartTime': '-PT1H',
   'DataAnalysisEndTime': '-PT0H'},
  'MonitoringJobDefinitionName': 'model-quality-job-definition-2026-02-22-04-52-01-434',
  'MonitoringType': 'ModelQuality'},
 'EndpointName': 'ccpp-xgb-model-monitor-2026-02-22-0147',
 'ResponseMetadata': {'RequestId': 'aac0ee0f-53b4-4395-b9d2-46602fdf2e9c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'aac0ee0f-53b4-4395-b9d2-46602fdf2e9c',
   '

In [176]:
#lightGBM_model_quality_monitor.delete_monitoring_schedule()

INFO:sagemaker:Deleting Monitoring Schedule with name: Xgb-model-monitoring-schedule-2026-02-22-04201771734032
INFO:sagemaker.model_monitor.model_monitoring:Deleting Model Quality Job Definition with name: model-quality-job-definition-2026-02-22-04-20-32-376


#### 4.5 Examine monitoring schedule executions

In [181]:
# Initially there will be no executions since the first execution happens at the top of the hour
executions = xgb_model_quality_monitor.list_executions()
executions

[]

In [182]:
# Wait for the first execution of the monitoring_schedule
print("Waiting for first execution ", end="")
while True:
    execution = xgb_model_quality_monitor.describe_schedule().get(
        "LastMonitoringExecutionSummary"
    )
    if execution:
        break
    print(".", end="", flush=True)
    sleep(10)
print()
print("Execution found!")

Waiting for first execution .....................................

/tmp/ipykernel_89118/258847449.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  upload_ground_truth(fake_records, datetime.utcnow())


Uploading 649 records to s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/ground_truth_data/2026-02-22-01-46-50/2026/02/22/04/5832.jsonl
....................
Execution found!


In [184]:
while not executions:
    executions = xgb_model_quality_monitor.list_executions()
    print(".", end="", flush=True)
    sleep(10)
latest_execution = executions[-1]
latest_execution.describe()

{'ProcessingInputs': [{'InputName': 'groundtruth_input_1',
   'AppManaged': False,
   'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/ground_truth_data/2026-02-22-01-46-50/2026/02/22/03',
    'LocalPath': '/opt/ml/processing/groundtruth/2026/02/22/03',
    'S3DataType': 'S3Prefix',
    'S3InputMode': 'File',
    'S3DataDistributionType': 'FullyReplicated',
    'S3CompressionType': 'None'}},
  {'InputName': 'endpoint_input_1',
   'AppManaged': False,
   'S3Input': {'S3Uri': 's3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/datacapture/ccpp-xgb-model-monitor-2026-02-22-0147/AllTraffic/2026/02/22/03',
    'LocalPath': '/opt/ml/processing/input_data/ccpp-xgb-model-monitor-2026-02-22-0147/AllTraffic/2026/02/22/03',
    'S3DataType': 'S3Prefix',
    'S3InputMode': 'File',
    'S3DataDistributionType': 'FullyReplicated',
    'S3CompressionType': 'None'}}],
 'ProcessingOutputConfig': {'Outputs': [{'O

In [185]:
try:
    status = execution["MonitoringExecutionStatus"]

    while status in ["Pending", "InProgress"]:
        print("Waiting for execution to finish", end="")
        latest_execution.wait(logs=False)
        latest_job = latest_execution.describe()
        print()
        print(f"{latest_job['ProcessingJobName']} job status:", latest_job["ProcessingJobStatus"])
        print(
            f"{latest_job['ProcessingJobName']} job exit message, if any:",
            latest_job.get("ExitMessage"),
        )
        print(
            f"{latest_job['ProcessingJobName']} job failure reason, if any:",
            latest_job.get("FailureReason"),
        )
        sleep(
            30
        )  # model quality executions consist of two Processing jobs, wait for second job to start
        latest_execution = xgb_model_quality_monitor.list_executions()[-1]
        execution = xgb_model_quality_monitor.describe_schedule()["LastMonitoringExecutionSummary"]
        status = execution["MonitoringExecutionStatus"]

    print("Execution status is:", status)

    if status != "Completed":
        print(execution)
        print(
            "====STOP==== \n No completed executions to inspect further. Please wait till an execution completes or investigate previously reported failures."
        )
except Exception as e:
    print(f"An error occurred: {e}")

Waiting for execution to finish..........................!
groundtruth-merge-202602220452-d5d7b84e3e39a3a46233d992 job status: Completed
groundtruth-merge-202602220452-d5d7b84e3e39a3a46233d992 job exit message, if any: None
groundtruth-merge-202602220452-d5d7b84e3e39a3a46233d992 job failure reason, if any: None
Waiting for execution to finish.....................................................!
model-quality-monitoring-202602220452-d5d7b84e3e39a3a46233d992 job status: Completed
model-quality-monitoring-202602220452-d5d7b84e3e39a3a46233d992 job exit message, if any: CompletedWithViolations: Job completed successfully with 4 violations.
model-quality-monitoring-202602220452-d5d7b84e3e39a3a46233d992 job failure reason, if any: None
Execution status is: CompletedWithViolations
{'MonitoringScheduleName': 'Xgb-model-monitoring-schedule-2026-02-22-04521771735921', 'ScheduledTime': datetime.datetime(2026, 2, 22, 4, 52, 6, tzinfo=tzlocal()), 'CreationTime': datetime.datetime(2026, 2, 22, 5, 1,

In [186]:
latest_execution = xgb_model_quality_monitor.list_executions()[-1]
report_uri = latest_execution.describe()["ProcessingOutputConfig"]["Outputs"][0]["S3Output"][
    "S3Uri"
]
print("Report Uri:", report_uri)

Report Uri: s3://sagemaker-us-east-1-582544415612/sagemaker/XGB-ModelQualityMonitor-20240225test/baselining/results/ccpp-xgb-model-monitor-2026-02-22-0147/Xgb-model-monitoring-schedule-2026-02-22-04521771735921/2026/02/22/04


#### 4.6 View violations generated by monitoring schedule

In [187]:
pd.options.display.max_colwidth = None
violations = latest_execution.constraint_violations().body_dict["violations"]
violations_df = pd.json_normalize(violations)
violations_df.head(10)

,constraint_check_type,description,metric_name
0,GreaterThanThreshold,Metric mae with 12.079317519741934 +/- 0.0049933948314038 was GreaterThanThreshold '1.3649406516611295',mae
1,GreaterThanThreshold,Metric mse with 147.63632716546823 +/- 0.10847248007089161 was GreaterThanThreshold '2.721507355004372',mse
2,GreaterThanThreshold,Metric rmse with 12.1505690058313 +/- 0.004464416858896037 was GreaterThanThreshold '1.649699171062522',rmse
3,LessThanThreshold,Metric r2 with 0.15365069829517952 +/- 0.006728626109969033 was LessThanThreshold '0.9868210200393686',r2


### Section 5 - Analyze model quality CloudWatch metrics <a id='analyze-cloudwatch-metrics'></a> 

#### 5.1 List the CW metrics generated.

In [188]:
# Create CloudWatch client
cw_client = boto3.Session().client("cloudwatch")

namespace = "aws/sagemaker/Endpoints/model-metrics"

cw_dimensions = [
    {"Name": "Endpoint", "Value": endpoint_name},
    {"Name": "MonitoringSchedule", "Value": smartgrid_monitor_schedule_name},
]

In [192]:
# List metrics through the pagination interface
paginator = cw_client.get_paginator("list_metrics")

for response in paginator.paginate(Dimensions=cw_dimensions, Namespace=namespace):
    model_quality_metrics = response["Metrics"]
    for metric in model_quality_metrics:
        print(metric["MetricName"])

total_number_of_violations
mae
rmse
mse
r2


#### 5.2 Create a CloudWatch Alarm for MAE

In [193]:
alarm_name = "MODEL_QUALITY_mae"
alarm_desc = (
    "Trigger an CloudWatch alarm when the mae drifts away from the baseline constraints"
)
mdoel_quality_mae_drift_threshold = (
    1.0  ##Setting this threshold purposefully low to see the alarm quickly.
)
metric_name = "mae"
namespace = "aws/sagemaker/Endpoints/model-metrics"

cw_client.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription=alarm_desc,
    ActionsEnabled=True,
    MetricName=metric_name,
    Namespace=namespace,
    Statistic="Average",
    Dimensions=[
        {"Name": "Endpoint", "Value": endpoint_name},
        {"Name": "MonitoringSchedule", "Value": smartgrid_monitor_schedule_name},
    ],
    Period=600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=mdoel_quality_mae_drift_threshold ,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="breaching",
)

{'ResponseMetadata': {'RequestId': 'da4f848c-cfcd-469a-a02e-06f6603fd40e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'da4f848c-cfcd-469a-a02e-06f6603fd40e',
   'content-type': 'application/x-amz-json-1.0',
   'content-length': '0',
   'date': 'Sun, 22 Feb 2026 05:24:58 GMT'},
  'RetryAttempts': 0}}

### Clean up

In [ ]:
xgb_model_quality_monitor.delete_monitoring_schedule()
sleep(60)  # actually wait for the deletion

In [ ]:
predictor.delete_model()
predictor.delete_endpoint()

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>